# Week 2 / Issue 2 - Download CFPB Complaint Data

This notebook downloads raw complaint records from the CFPB Consumer Complaint Database API for the Financial Complaint Auto-Routing with NLP project.

Goal:
- Download calendar year 2024 records that include public consumer complaint narratives.
- Text input for later modeling: `complaint_what_happened`
- Target label for later modeling: `product`
- Raw local file: `data/raw/cfpb_complaints_2024_raw.csv`

Other CFPB columns are kept in the raw file for traceability. Later modeling should avoid leakage by mainly using `complaint_what_happened` as the model feature and `product` as the supervised multi-class target.

Scope: this notebook only downloads, validates, saves, and verifies raw ingest data. Cleaning, EDA, feature engineering, and modeling belong in later notebooks.

Git safety: the raw CSV stays local under `data/raw/`. The raw data folder contents and `*.csv` files are ignored by `.gitignore`, so this file should not be committed or uploaded to GitHub.


## 1.Imports and Settings


In [ ]:
from pathlib import Path
import time

import pandas as pd
import requests

BASE_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
REQUEST_HEADERS = {"User-Agent": "financial-complaint-auto-routing-nlp/1.0 (student ML project)"}

# Save to the project data/raw folder whether the notebook is run from
# notebooks/ or from the project root in VS Code.
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RAW_DIR / "cfpb_complaints_2024_raw.csv"

DATE_START = "2024-01-01"
DATE_END_EXCLUSIVE = "2025-01-01"
API_DATE_END = "2024-12-31"  # Avoid pulling 2025-01-01 rows from the API.

# Use 1_000 for a quick smoke test; use 50_000 for the final Week 2 download.
TARGET_ROWS = 50_000

PAGE_SIZE = 100  # CFPB API maximum page size.
MAX_PAGES = 1_000
MAX_PAGES_WITHOUT_NEW_VALID_ROWS = 10
SLEEP_SECONDS = 0.2


## 2.Helper Functions


In [ ]:
def extract_hits(api_json):
    """Return the list of hit objects from a CFPB API response."""
    return api_json.get("hits", {}).get("hits", [])


def extract_records(api_json):
    """Extract complaint records from CFPB API JSON response."""
    return [hit.get("_source", {}) for hit in extract_hits(api_json)]


def get_next_search_after(api_json):
    """Build the search_after token from the final hit in the current page."""
    hits = extract_hits(api_json)

    if not hits:
        return None

    sort_values = hits[-1].get("sort")

    if not sort_values:
        return None

    return "_".join(str(value) for value in sort_values)


def count_valid_2024_records(records):
    """Count records that pass the required 2024 raw-data checks."""
    if not records:
        return 0

    temp_df = pd.DataFrame(records)
    required_columns = ["complaint_what_happened", "product", "date_received", "complaint_id"]

    if any(col not in temp_df.columns for col in required_columns):
        return 0

    date_start = pd.Timestamp(DATE_START, tz="UTC")
    date_end_exclusive = pd.Timestamp(DATE_END_EXCLUSIVE, tz="UTC")
    date_received_dt = pd.to_datetime(temp_df["date_received"], errors="coerce", utc=True)

    has_narrative = temp_df["complaint_what_happened"].fillna("").astype(str).str.strip().ne("")
    has_product = temp_df["product"].fillna("").astype(str).str.strip().ne("")
    has_complaint_id = temp_df["complaint_id"].fillna("").astype(str).str.strip().ne("")
    in_2024 = date_received_dt.notna() & date_received_dt.ge(date_start) & date_received_dt.lt(date_end_exclusive)

    valid_df = temp_df[has_narrative & has_product & has_complaint_id & in_2024]
    return valid_df.drop_duplicates(subset=["complaint_id"]).shape[0]


def fetch_page(search_after=None):
    """Fetch one page of CFPB complaint records with public narratives."""
    params = {
        "date_received_min": DATE_START,
        "date_received_max": API_DATE_END,
        "has_narrative": "true",
        "sort": "created_date_desc",
        "size": PAGE_SIZE,
        "frm": 0,
        "no_aggs": "true",
        "no_highlight": "true",
    }

    if search_after is not None:
        params["search_after"] = search_after

    response = requests.get(BASE_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


## 3.Test the API First


In [ ]:
test_json = fetch_page()
test_records = extract_records(test_json)

if not test_records:
    raise RuntimeError("No records returned. Check the API connection and query parameters.")

total_available = test_json.get("hits", {}).get("total", {}).get("value")
first_record = test_records[0]
required_columns = ["complaint_what_happened", "product", "date_received", "complaint_id"]
missing_columns = [col for col in required_columns if col not in first_record]

print(f"API date request: {DATE_START} through {API_DATE_END}")
print(f"Local validation window: {DATE_START} to before {DATE_END_EXCLUSIVE}")
print(f"Total matching API records: {total_available:,}")
print(f"Returned test records: {len(test_records)}")
print("Required columns present:", missing_columns == [])
print("First record has narrative:", bool(str(first_record.get("complaint_what_happened", "")).strip()))
print("First record has product:", bool(str(first_record.get("product", "")).strip()))
print("Example keys:")
print(sorted(first_record.keys()))

if missing_columns:
    raise ValueError(f"Missing required columns in API response: {missing_columns}")


## 4.Download Raw Records


In [ ]:
all_records = []
search_after = None
page_number = 1
valid_2024_records_collected = 0
pages_without_new_valid_rows = 0

while valid_2024_records_collected < TARGET_ROWS and page_number <= MAX_PAGES:
    previous_valid_count = valid_2024_records_collected
    api_json = fetch_page(search_after=search_after)
    batch = extract_records(api_json)

    if not batch:
        print("No more records returned.")
        break

    all_records.extend(batch)
    valid_2024_records_collected = count_valid_2024_records(all_records)

    if valid_2024_records_collected == previous_valid_count:
        pages_without_new_valid_rows += 1
    else:
        pages_without_new_valid_rows = 0

    print(f"Page {page_number}: downloaded raw records: {len(all_records):,}")
    print(f"Page {page_number}: valid 2024 records collected: {valid_2024_records_collected:,} of {TARGET_ROWS:,}")

    if pages_without_new_valid_rows >= MAX_PAGES_WITHOUT_NEW_VALID_ROWS:
        raise RuntimeError(
            "Stopped because multiple API pages added no new valid 2024 records. "
            "Check API date parameters and pagination."
        )

    search_after = get_next_search_after(api_json)

    if search_after is None:
        print("No search_after token found. Stopping pagination.")
        break

    page_number += 1
    time.sleep(SLEEP_SECONDS)

if valid_2024_records_collected < TARGET_ROWS:
    raise RuntimeError(
        f"Only collected {valid_2024_records_collected:,} valid 2024 records; "
        f"target is {TARGET_ROWS:,}."
    )

print(f"Finished download. Total raw records collected: {len(all_records):,}")
print(f"Finished download. Valid 2024 records collected: {valid_2024_records_collected:,}")


## 5.Convert to DataFrame and Check Required Fields


In [ ]:
df = pd.DataFrame(all_records)

print("Raw shape before filtering:", df.shape)
print("Columns:")
print(sorted(df.columns.tolist()))

required_columns = ["complaint_what_happened", "product", "date_received", "complaint_id"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

rows_before_required_filter = len(df)

text_label_columns = ["complaint_what_happened", "product"]

df = df.dropna(subset=text_label_columns).copy()
df["complaint_what_happened"] = df["complaint_what_happened"].astype(str).str.strip()
df["product"] = df["product"].astype(str).str.strip()
df = df[(df["complaint_what_happened"] != "") & (df["product"] != "")].copy()

rows_removed_required = rows_before_required_filter - len(df)
print(f"Rows removed for missing or empty narratives/products: {rows_removed_required:,}")

date_start = pd.Timestamp(DATE_START, tz="UTC")
date_end_exclusive = pd.Timestamp(DATE_END_EXCLUSIVE, tz="UTC")

rows_before_date_filter = len(df)
date_received_dt = pd.to_datetime(df["date_received"], errors="coerce", utc=True)
in_date_range = date_received_dt.notna() & date_received_dt.ge(date_start) & date_received_dt.lt(date_end_exclusive)
df = df[in_date_range].copy()

rows_removed_outside_2024 = rows_before_date_filter - len(df)
print(f"Rows removed outside 2024 date range: {rows_removed_outside_2024:,}")

empty_complaint_ids = df["complaint_id"].fillna("").astype(str).str.strip().eq("").sum()

if empty_complaint_ids > 0:
    raise ValueError(f"Found {empty_complaint_ids:,} rows with missing complaint_id values.")

duplicate_rows_removed = 0

rows_before_dedup = len(df)
df = df.drop_duplicates(subset=["complaint_id"]).copy()
duplicate_rows_removed = rows_before_dedup - len(df)

print(f"Duplicate complaint_id rows removed: {duplicate_rows_removed:,}")

if df.empty:
    raise ValueError("No rows remain after required-field and 2024 date filtering.")

if len(df) > TARGET_ROWS:
    df = df.head(TARGET_ROWS).copy()

final_date_received_dt = pd.to_datetime(df["date_received"], errors="coerce", utc=True)
final_date_min = final_date_received_dt.min()
final_date_max = final_date_received_dt.max()

assert final_date_received_dt.notna().all()
assert final_date_received_dt.ge(date_start).all()
assert final_date_received_dt.lt(date_end_exclusive).all()

print("Final shape:", df.shape)
print(f"Final date range: {final_date_min.date()} to {final_date_max.date()}")
print("Product label distribution:")
print(df["product"].value_counts().head(20))
print("Number of product classes:", df["product"].nunique())


## 6.Save Raw CSV Locally


In [ ]:
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Saved raw data to: {OUTPUT_PATH}")
print(f"Absolute path: {OUTPUT_PATH.resolve()}")
print(f"Rows saved: {len(df):,}")


## 7.Verify Saved File


In [ ]:
if not OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Expected output file was not created: {OUTPUT_PATH}")

saved_df = pd.read_csv(OUTPUT_PATH)

missing_saved_columns = [col for col in required_columns if col not in saved_df.columns]

if missing_saved_columns:
    raise ValueError(f"Saved CSV is missing required columns: {missing_saved_columns}")

empty_saved_narratives = saved_df["complaint_what_happened"].fillna("").astype(str).str.strip().eq("").sum()
empty_saved_products = saved_df["product"].fillna("").astype(str).str.strip().eq("").sum()
empty_saved_complaint_ids = saved_df["complaint_id"].fillna("").astype(str).str.strip().eq("").sum()

date_start = pd.Timestamp(DATE_START, tz="UTC")
date_end_exclusive = pd.Timestamp(DATE_END_EXCLUSIVE, tz="UTC")

saved_date_received_dt = pd.to_datetime(saved_df["date_received"], errors="coerce", utc=True)
saved_in_date_range = saved_date_received_dt.notna() & saved_date_received_dt.ge(date_start) & saved_date_received_dt.lt(date_end_exclusive)
saved_rows_outside_2024 = (~saved_in_date_range).sum()

print("Saved file shape:", saved_df.shape)
print("Missing or empty narratives:", empty_saved_narratives)
print("Missing or empty product labels:", empty_saved_products)
print("Missing or empty complaint_id values:", empty_saved_complaint_ids)
print("Rows outside 2024 date range:", saved_rows_outside_2024)
print(f"Saved date range: {saved_date_received_dt.min().date()} to {saved_date_received_dt.max().date()}")
print("Number of product classes:", saved_df["product"].nunique())

duplicate_saved_ids = saved_df["complaint_id"].duplicated().sum()
print("Duplicate complaint_id values:", duplicate_saved_ids)

assert len(saved_df) == len(df)
assert empty_saved_narratives == 0
assert empty_saved_products == 0
assert empty_saved_complaint_ids == 0
assert duplicate_saved_ids == 0
assert saved_rows_outside_2024 == 0

print("Saved CSV verification passed.")


## Week 2 Download Summary

This notebook completed the raw data download step for the **Financial Complaint Auto-Routing with NLP** project.

Summary:

* Downloaded **50,000 CFPB complaint records** with public consumer complaint narratives.
* Saved the raw CSV locally as `data/raw/cfpb_complaints_2024_raw.csv`.
* Confirmed the raw CSV contains the required input text column: `complaint_what_happened`.
* Confirmed the raw CSV contains the target label column: `product`.
* Verified there are no missing or empty complaint narratives.
* Verified there are no missing or empty product labels.
* Verified there are no duplicate `complaint_id` values.
* Verified all saved records are inside calendar year 2024.
* Kept all CFPB API columns in the raw CSV for traceability.
* Confirmed the raw CSV is ignored by Git and should not be uploaded to GitHub.

Final local validation:

* Shape: **50,000 rows × 17 columns**
* Date range: **2024-12-11 to 2024-12-31**
* Rows outside 2024: **0**
* Product classes: **11**

Note: The API returns records sorted newest first, so this 50,000-row dataset comes from late 2024 records. Later modeling notebooks will use `complaint_what_happened` as the text feature and `product` as the supervised classification label.
